#  Real-Time Intelligence — Complete Reference (`ws_mch_iot`)

**Machinery IoT — hot path (Flow A).** Live machine telemetry → classification → real-time
aggregates → alerts → operations dashboard. Everything in this folder, end to end, nothing left out.

```
Simulator ──SAS──▶ es_mch (Eventstream) ──DirectIngestion──▶ kdb_mch.tbl_mch_raw
                                                                   │ update policy fn_mch_clean()
                                                                   ▼
                                                             tbl_mch_clean ──▶ materialized views ──▶ dsh_mch (dashboard)
                                                                   │                                 ▲
                                                                   └─▶ fn_mch_alerts / errors / status_alerts ─▶ act_mch_critical (email)
```

| Item | Type | Role |
|---|---|---|
| `eh_mch` | Eventhouse | hot store (KQL engine) |
| `kdb_mch` | KQL Database | tables, policies, MVs, functions |
| `es_mch` | Eventstream | ingestion (custom endpoint → tbl_mch_raw) |
| `qs_mch` | KQL Queryset | all KQL, authored as named queries |
| `dsh_mch` | KQL Dashboard | alert-operations dashboard (10 tiles) |
| `act_mch_critical` | Activator/Reflex | email alerts on critical + error + status feeds |


## 1. Ingestion — `es_mch` Eventstream

- **Source** `src_sim` — **Custom Endpoint** (Event Hub-compatible). No external secret needed; the
  endpoint mints its own SAS. Fetch the connection string:
  `GET workspaces/{ws}/eventstreams/{es}/sources/{srcId}/connection` (rotates on each rebuild).
- **Destination** `Eventhouse` — **Direct ingestion** into **`kdb_mch` / `tbl_mch_raw`**, JSON/UTF-8.
- **Topology** `src_sim → es_mch-stream → tbl_mch_raw`. Built and **published in the portal** (the
  destination's identity/permission binding is portal-gated; the topology JSON is API-definable).
- **Producer** — `sim_mch.py` (local, `azure-eventhub`) and `nb_mch_stream_check` (in-Fabric). Both
  publish the packet below; the roster (`roster.py`) is shared with the dimensions so ids always match.

**Event packet (thin — quality is derived downstream, not in the payload):**
```json
{ "event_id":"uuid", "event_ts":"2026-06-20T10:00:00Z",
  "plant_id":"PLANT_01", "line_id":"LINE_A1", "machine_id":"MACH_1001", "status":"RUNNING",
  "temperature":78.5, "vibration":0.8, "pressure":6.2, "rpm":1200.0, "power_kw":45.3,
  "schema_version":"1.0" }
```
`status ∈ {RUNNING, IDLE, MAINTENANCE}`. `rpm` is emitted as a float so the Eventstream infers `real`
(matches the table column — no Int64→real coercion warning).


## 2. Tables (`kdb_mch`)

### `tbl_mch_raw` — streaming landing (append-only, replay/backfill source)
`event_id:string, event_ts:datetime, plant_id:string, line_id:string, machine_id:string,
status:string, temperature:real, vibration:real, pressure:real, rpm:real, power_kw:real, schema_version:string`

### `tbl_mch_clean` — cleansed + classified (the analytic table)
adds: `ingest_ts:datetime, lag_sec:real, quality_flag:string, is_alert:bool, is_anomaly:bool,
is_late:bool, dq_reason:string`

**Ingestion mode:** streaming ingestion **ON** (Eventstream needs it) + 30 s batching.

## 3. Update policy — `fn_mch_clean()`  (raw → clean, automatic)
Runs on every ingest into `tbl_mch_raw`; output lands in `tbl_mch_clean`.

- `ingest_ts = ingestion_time()`, `lag_sec = ingest_ts − event_ts`, `is_late = lag_sec > 300`
  → **mistimed / out-of-order packet detection**.
- **Quality taxonomy** (`quality_flag`):

| Flag | Meaning | Rule |
|---|---|---|
| `NULL` | missing sensor field | any sensor null |
| `BAD` | physically impossible | temp<0/>200, rpm<0/>5000, etc. |
| `ANOMALY` | severe breach | temp>105 ∨ vib>7.1 ∨ rpm>1800 ∨ power>95 ∨ pressure>10 |
| `ALERT` | warning breach | temp>85 ∨ vib>2.8 ∨ rpm>1500 ∨ power>75 ∨ pressure∉[4,8] |
| `NORMAL` | in band | otherwise |

`is_alert`/`is_anomaly` are booleans for cheap filtering; `dq_reason` records *why* (NULL_FIELD /
OUT_OF_RANGE / LATE_EVENT_<n>s).


## 4. Materialized views (pre-aggregated, power the dashboard cheaply)

| MV | Grain | Columns |
|---|---|---|
| `mv_mch_machine_min` | machine × 1 min | events, alerts, anomalies, late, avg_temp, avg_vib, avg_rpm, max_lag_sec |
| `mv_mch_status` | status × quality × 1 min | events |
| `mv_mch_alerts` | machine × 1 min (alert/anomaly only) | alerts, anomalies, max_temp, max_vib |
| `mv_mch_status_health` | machine × status_issue × 1 min | issues |

## 5. Functions (alert feeds + utilities)

| Function | Purpose | Consumer |
|---|---|---|
| `fn_mch_alerts()` | critical packets (ALERT ∨ ANOMALY) + reason | Activator rule **alerts**, dashboard |
| `fn_mch_errors()` | DQ errors (BAD/NULL) + late → BAD_DATA/MISSING_FIELD/LATE_PACKET | Activator rule **erros**, dashboard |
| `fn_mch_late()` | late/out-of-order packets ranked by lag | dashboard / triage |
| `fn_mch_status_anomalies()` | status-vs-data mismatches (see §6) | dashboard, status feed |
| `fn_mch_status_alerts()` | status violations for alerting | recommended 3rd Activator rule |


## 6. Status-aware semantics (the *meaning* in the data)

The telemetry must be consistent with the machine's declared `status`:

| Status | Expected | Violation | Why it matters |
|---|---|---|---|
| `RUNNING` | production bands | (covered by quality taxonomy) | normal ops |
| `IDLE` | rpm≈0, low power | **`IDLE_PHANTOM_LOAD`** — rpm>150 ∨ power>15 | energy waste / control fault |
| `MAINTENANCE` | locked out (~0) | **`MAINTENANCE_LOCKOUT_VIOLATION`** — rpm>50 ∨ temp>45 ∨ power>10 | **SAFETY** — machine running during lockout |

Detected by `fn_mch_status_anomalies()`, rolled up in `mv_mch_status_health`, alertable via
`fn_mch_status_alerts()`. The simulator injects these violations (~7 % of IDLE, ~20 % of MAINTENANCE).

## 7. Retention / caching (cost tiering)
| Table | Hot cache | Retain |
|---|---|---|
| `tbl_mch_clean` | 7 days | 90 days |
| `tbl_mch_raw` | 1 day | 7 days |


## 8. Activator — `act_mch_critical` (email alerts)

Two rules are live (built in the rule-builder; the rule/action schema is portal-gated):

| Rule | Source query | Condition | Action |
|---|---|---|---|
| **alerts** | `fn_mch_alerts() \| where event_ts > ago(5m)` | OnEveryValue | Email → anujfabric@…, anuj.shah@simformsolutions.com |
| **erros** | `fn_mch_errors() \| where event_ts > ago(5m)` | OnEveryValue | Email → same recipients |

> **Recommended 3rd rule (status):** Set alert on `fn_mch_status_alerts() | where event_ts > ago(5m)`
> → email; `MAINTENANCE_LOCKOUT_VIOLATION` is safety-critical.

## 9. Dashboard — `dsh_mch` ("Machinery IoT — Alert Operations", 10 tiles)
KPIs · alert/anomaly trend · quality donut · top offending machines · peak temperature · status mix ·
live critical feed · DQ errors · **status violations** · **lockout/phantom feed**. Time-range picker
default 4 h; all tiles read `mv_mch_*` / `fn_mch_*`.

## 10. Eventhouse security
- `fn_mch_rls()` — KQL row-level security pattern: a plant-scoped group sees only its plant; owner/
  services see all. **Enabling on `tbl_mch_clean` is blocked because the table has materialized views**
  (KQL limitation — MVs require an open base table). Function is provided as the ready pattern.

## 11. Operate
```bash
python sim_mch.py --loop --rate 12          # stream continuously
python sim_mch.py --count 600               # one batch
# or run notebook nb_mch_stream_check (in-Fabric, streams via SAS + verifies)
```
See the warehouse doc for the dimension/master-data side.
